In [16]:
import platform

print("Операционная система:", platform.system())

Операционная система: Linux


In [17]:

%pip install -q -r requirements.txt


In [18]:
import os
import re
import json
import math
import random
import shutil
import gc

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm
from google.colab import files, drive
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

In [19]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "jinaai/jina-reranker-v2-base-multilingual"

MAX_LENGTH = 384
MAX_TEXT_CHARS = 2400

BATCH_SIZE = 2
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 8

EPOCHS = 2
LEARNING_RATE = 5e-5
SAVE_EVERY = 100

device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))


GPU: Tesla T4


In [20]:
SAVE_DIR = Path("jina_final")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = SAVE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

BEST_DIR = SAVE_DIR / "best_adapter"


In [21]:
train = pd.read_csv(
    "train.csv",
    dtype={"id": str, "query_id": str}
)

test = pd.read_csv(
    "test.csv",
    dtype={"id": str, "query_id": str}
)

sample = pd.read_csv(
    "sample_submission.csv",
    dtype={"id": str}
)

In [22]:
print("Train:", train.shape)
print("Test:", test.shape)
print("Sample:", sample.shape)

Train: (31530, 12)
Test: (13507, 11)
Sample: (13507, 2)


In [23]:
def prepare_data(df):
    df = df.copy()

    for col in ["query", "title", "text"]:
        df[col] = (
            df[col].fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    df["document"] = (
        "Заголовок: " + df["title"] +
        "\nТекст: " + df["text"]
    ).str.slice(0, MAX_TEXT_CHARS)

    return df


train = prepare_data(train)
test = prepare_data(test)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(train, groups=train["query_id"])
)

tr = train.iloc[train_idx].reset_index(drop=True)
val = train.iloc[val_idx].reset_index(drop=True)

print("Train:", len(tr), "Validation:", len(val))


Train: 25193 Validation: 6337


In [24]:

import torch.nn as nn

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_flash_attn=False,
    torch_dtype=torch.float32
)

target_modules = [
    name
    for name, layer in base_model.named_modules()
    if isinstance(layer, nn.Linear)
    and name.endswith(".mixer.out_proj")
    and "classifier" not in name
]

print("Найдено слоёв для LoRA:", len(target_modules))

if not target_modules:
    raise ValueError(
        "Слои внимания не найдены"
    )

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=target_modules,
    modules_to_save=["classifier"],
    bias="none"
)


model = get_peft_model(base_model, lora_config)
model = model.to(device)

model.print_trainable_parameters()




Найдено слоёв для LoRA: 12
trainable params: 738,817 || all params: 279,176,450 || trainable%: 0.2646


In [25]:

class RankingDataset(Dataset):
    def __init__(self, df):
        self.labels = (
            df["relevance"].to_numpy(dtype=np.float32)
            if "relevance" in df.columns else None
        )

        self.data = []

        for i in tqdm(
            range(0, len(df), 256),
            desc="Токенизация"
        ):
            batch = df.iloc[i:i+256]

            enc = tokenizer(
                batch["query"].tolist(),
                batch["document"].tolist(),
                truncation=True,
                max_length=MAX_LENGTH
            )

            for j in range(len(batch)):
                self.data.append({
                    k: v[j] for k, v in enc.items()
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i].copy()

        if self.labels is not None:
            item["labels"] = float(self.labels[i])

        return item


def collate(batch):
    labels = None

    if "labels" in batch[0]:
        labels = torch.tensor(
            [x["labels"] for x in batch],
            dtype=torch.float32
        )

    inputs = tokenizer.pad(
        [
            {k: v for k, v in x.items() if k != "labels"}
            for x in batch
        ],
        padding=True,
        return_tensors="pt"
    )

    if labels is not None:
        inputs["labels"] = labels

    return inputs


train_ds = RankingDataset(tr)
val_ds = RankingDataset(val)

val_loader = DataLoader(
    val_ds,
    batch_size=EVAL_BATCH_SIZE,
    collate_fn=collate,
    pin_memory=True
)


Токенизация:   0%|          | 0/99 [00:00<?, ?it/s]

Токенизация:   0%|          | 0/25 [00:00<?, ?it/s]

In [26]:

def ndcg_at_10(df, predictions):
    data = df[["query_id", "id", "relevance"]].copy()
    data["prediction"] = predictions

    scores = []

    for _, group in data.groupby("query_id"):
        ranked = group.sort_values(
            ["prediction", "id"],
            ascending=[False, True]
        )

        rel = ranked["relevance"].to_numpy()[:10]
        ideal = np.sort(group["relevance"].to_numpy())[::-1][:10]

        discount = 1 / np.log2(np.arange(2, len(rel) + 2))

        dcg = np.sum(rel * discount)
        idcg = np.sum(ideal * discount[:len(ideal)])

        scores.append(dcg / idcg if idcg > 0 else 0)

    return np.mean(scores)


@torch.inference_mode()
def predict(model, loader, desc="Предсказания"):
    model.eval()
    scores = []

    for batch in tqdm(loader, desc=desc):
        inputs = {
            k: v.to(device)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits

        scores.extend(logits.float().view(-1).cpu().tolist())

    return np.array(scores)


In [27]:

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01
)

STEPS_PER_EPOCH = math.ceil(
    math.ceil(len(train_ds) / BATCH_SIZE) / GRAD_ACCUM_STEPS
)

TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(TOTAL_STEPS * 0.1),
    num_training_steps=TOTAL_STEPS
)

scaler = torch.amp.GradScaler("cuda")
loss_fn = torch.nn.BCEWithLogitsLoss()


In [28]:

def save_checkpoint(epoch, batch_num, global_step):
    path = CHECKPOINT_DIR / f"step_{global_step}"
    path.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(path)

    torch.save({
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "batch_num": batch_num,
        "global_step": global_step
    }, path / "training_state.pt")

    checkpoints = sorted(
        CHECKPOINT_DIR.glob("step_*"),
        key=lambda p: int(p.name.split("_")[-1])
    )

    for old in checkpoints[:-2]:
        shutil.rmtree(old)


def load_checkpoint():
    checkpoints = [
        p for p in CHECKPOINT_DIR.glob("step_*")
        if (p / "training_state.pt").exists()
        and (p / "adapter_config.json").exists()
    ]

    if not checkpoints:
        return 0, 0, 0

    path = max(
        checkpoints,
        key=lambda p: int(p.name.split("_")[-1])
    )

    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file

    weights = load_file(str(path / "adapter_model.safetensors"))
    set_peft_model_state_dict(model, weights)

    state = torch.load(
        path / "training_state.pt",
        map_location="cpu",
        weights_only=False
    )

    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])

    return (
        state["epoch"],
        state["batch_num"],
        state["global_step"]
    )


start_epoch, start_batch, global_step = load_checkpoint()


In [ ]:

best_score = -1.0

result_file = SAVE_DIR / "best_result.json"

if result_file.exists():
    with open(result_file) as f:
        best_score = json.load(f)["ndcg"]

for epoch in range(start_epoch, EPOCHS):

    generator = torch.Generator().manual_seed(SEED + epoch)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=collate,
        pin_memory=True
    )

    model.train()
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        enumerate(train_loader, start=1),
        total=len(train_loader),
        desc=f"Эпоха {epoch + 1}/{EPOCHS}",
        unit="batch"
    )

    for batch_num, batch in progress:

        if epoch == start_epoch and batch_num <= start_batch:
            continue

        labels = batch.pop("labels").to(device)

        inputs = {
            k: v.to(device)
            for k, v in batch.items()
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits.float().view(-1)
            loss = loss_fn(logits, labels)

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()

        if (
            batch_num % GRAD_ACCUM_STEPS == 0
            or batch_num == len(train_loader)
        ):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            if scaler.get_scale() >= old_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % SAVE_EVERY == 0:
                save_checkpoint(epoch, batch_num, global_step)

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    val_predictions = predict(
        model, val_loader, "Validation"
    )

    score = ndcg_at_10(val, val_predictions)
    print(f"NDCG@10: {score:.6f}")

    if score > best_score:
        best_score = float(score)

        model.save_pretrained(BEST_DIR)
        tokenizer.save_pretrained(BEST_DIR)

        with open(result_file, "w") as f:
            json.dump({
                "epoch": epoch + 1,
                "ndcg": best_score,
                "model": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "max_text_chars": MAX_TEXT_CHARS
            }, f, indent=2)

    save_checkpoint(epoch + 1, 0, global_step)

    start_batch = 0

print("Лучший NDCG@10:", best_score)

In [ ]:

# Освобождаем память GPU
del model, optimizer, scheduler, scaler
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_flash_attn=False,
    torch_dtype=torch.float32
)

model = PeftModel.from_pretrained(
    base_model,
    str(BEST_DIR),
    is_trainable=False
)

model = model.to(device)
model.eval()


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): XLMRobertaForSequenceClassification(
      (roberta): XLMRobertaModel(
        (embeddings): XLMRobertaEmbeddings(
          (word_embeddings): Embedding(250002, 768, padding_idx=1)
          (position_embeddings): Embedding(1026, 768)
          (token_type_embeddings): Embedding(1, 768)
        )
        (emb_drop): Dropout(p=0.1, inplace=False)
        (emb_ln): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (encoder): XLMRobertaEncoder(
          (layers): ModuleList(
            (0-11): 12 x Block(
              (mixer): MHA(
                (Wqkv): LinearResidual(in_features=768, out_features=2304, bias=True)
                (inner_attn): SelfAttention(
                  (drop): Dropout(p=0.1, inplace=False)
                )
                (inner_cross_attn): CrossAttention(
                  (drop): Dropout(p=0.1, inplace=False)
                )
                (out_proj): lor

In [ ]:

test_ds = RankingDataset(test)

test_loader = DataLoader(
    test_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate,
    pin_memory=True
)

test_predictions = predict(
    model,
    test_loader,
    "Предсказания test"
)

assert len(test_predictions) == len(test)
assert np.isfinite(test_predictions).all()


Токенизация:   0%|          | 0/53 [00:00<?, ?it/s]

Предсказания test:   0%|          | 0/1689 [00:00<?, ?it/s]

In [ ]:

predictions = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "prediction": test_predictions
})

submission = sample[["id"]].merge(
    predictions,
    on="id",
    how="left",
    validate="one_to_one"
)

assert submission["prediction"].notna().all()

submission.to_csv("submission.csv", index=False)

files.download("submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>